# StereoLite v8 — Kaggle Training + Multi-format Export

Trains the v8 architecture (MobileNetV2-100 ImageNet-pretrained encoder, HITNet-style tile hypothesis + RAFT-style iterative refinement, convex-upsample output head) on Scene Flow Driving using 2x T4 GPUs via DistributedDataParallel, then exports the best checkpoint to:

- **PyTorch state_dict** `.pth` — **primary GPU-inference format.** Load with `torch.load()` + `model.load_state_dict()` after importing `StereoLite`. Bundled with the model source code so it's self-contained.
- **TorchScript** `.ts.pt` — **also PyTorch-runnable on GPU**, but self-contained (no source code import needed). Load with `torch.jit.load('stereolite_v8.ts.pt').cuda()` and call like any nn.Module. Useful for C++/LibTorch deployment.
- **ONNX** `.onnx` — for benchmarking only (fp32, fp16, int8 variants produced via opset 17 + onnxruntime dynamic quantization).

## Setup before running

1. **Accelerator**: Settings > Accelerator > **GPU T4 x2** (the notebook also works on a single P100 — batch size auto-adjusts).
2. **Dataset**: Add Scene Flow Driving as a Kaggle Dataset input. The notebook auto-detects the two tarball-extracted subfolders (`driving__frames_finalpass/frames_finalpass/...` and `driving__disparity/disparity/...`) and stitches them into a loader-friendly layout at `/kaggle/working/sceneflow_driving/` using symlinks.
3. **Time budget**: one full 30-epoch run is ~2-3 h on T4 x2. Kaggle's 9-hour GPU session easily covers training + export + validation.

## Outputs

All artefacts land in `/kaggle/working/` and are zipped at the end as `stereolite_v8_artifacts.zip` for one-click download.


## 1. Environment check

In [ ]:
import torch, sys, os
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, ' CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
N = torch.cuda.device_count()
print('GPU count:', N)
for i in range(N):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')

assert N >= 1, 'No GPU detected — enable T4 x2 or P100 in Settings.'


## 2. Install missing dependencies

Most of what we need is pre-installed on Kaggle. We add `onnx`, ONNX Runtime and the fp16 converter.

In [ ]:
!pip install -q timm==1.0.26 onnx onnxruntime-gpu onnxconverter-common onnxscript
!pip list 2>/dev/null | grep -E '^(torch|timm|onnx)' | sort


## 2b. Stitch attached dataset into the loader-friendly layout

The Scene Flow Driving Kaggle dataset (uploaded as two tarballs) is extracted by Kaggle into two sibling folders:

```
/kaggle/input/.../sceneflow-driving/driving__frames_finalpass/frames_finalpass/...
/kaggle/input/.../sceneflow-driving/driving__disparity/disparity/...
```

Our loader expects `frames_finalpass/` and `disparity/` as siblings under one root. This cell creates symlinks at `/kaggle/working/sceneflow_driving/` that point at the right subdirectories under `/kaggle/input/`. No data is copied; symlinks are essentially free.


In [ ]:
import os, glob

STAGE = '/kaggle/working/sceneflow_driving'
os.makedirs(STAGE, exist_ok=True)

# Locate the two extracted folders under /kaggle/input/ (any depth).
frames_hits = glob.glob('/kaggle/input/**/frames_finalpass', recursive=True)
disp_hits = glob.glob('/kaggle/input/**/disparity', recursive=True)
if not frames_hits or not disp_hits:
    print('Could not auto-locate frames_finalpass/ or disparity/ under '
          '/kaggle/input/. Tree (depth <= 4):')
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        depth = root.replace('/kaggle/input', '').count('/')
        if depth <= 4:
            print(' ', root)
        if depth >= 4:
            dirs[:] = []
    raise FileNotFoundError('Scene Flow subfolders not found under /kaggle/input/')

# Pick the deepest hit (handles nested duplicates).
frames_src = sorted(frames_hits, key=len)[-1]
disp_src = sorted(disp_hits, key=len)[-1]
print(f'frames source: {frames_src}')
print(f'disp   source: {disp_src}')

for src, name in [(frames_src, 'frames_finalpass'), (disp_src, 'disparity')]:
    link = os.path.join(STAGE, name)
    if os.path.islink(link) or os.path.exists(link):
        if os.path.islink(link):
            os.remove(link)
        elif os.path.isdir(link):
            import shutil; shutil.rmtree(link)
    os.symlink(src, link)
    print(f'  linked {name} -> {src}')

print(f'\nDATA_ROOT = {STAGE}')
print('\n-- contents --')
for entry in sorted(os.listdir(STAGE)):
    p = os.path.join(STAGE, entry)
    real = os.path.realpath(p)
    n = sum(1 for _ in os.walk(p, followlinks=True))
    print(f'  {entry} -> {real} ({n} subdirs reachable)')


## 3. Config

Edit `DATA_ROOT` if auto-detect misses. Default assumes you ran the download cell above (which writes under `/kaggle/working/sceneflow_driving`), or attached a Kaggle Dataset under `/kaggle/input/`.

In [ ]:
import os

# Cell 2b created /kaggle/working/sceneflow_driving/{frames_finalpass,disparity}
# as symlinks into /kaggle/input/. Override DATA_ROOT here only if you
# attached a dataset that already has the loader-friendly layout.
DATA_ROOT = '/kaggle/working/sceneflow_driving'

EPOCHS        = 30
BATCH_PER_GPU = 12      # ~12.5 GB on a T4 at 512x832; drop to 8 if OOM
INF_H         = 512
INF_W         = 832
LR_PEAK       = 8e-4
RANDOM_ERASE  = 0.3

for sub in ('frames_finalpass', 'disparity'):
    p = os.path.join(DATA_ROOT, sub)
    if not os.path.isdir(p):
        raise FileNotFoundError(
            f'{p} not found. Did Cell 2b run successfully?')

print(f'DATA_ROOT = {DATA_ROOT}')
print(f'epochs={EPOCHS}  batch/GPU={BATCH_PER_GPU}  input={INF_H}x{INF_W}')
print(f'peak LR={LR_PEAK}  random-erase p={RANDOM_ERASE}')


## 4. Write model source files

We recreate the exact repo layout under `/kaggle/working/src/` so the training script can `import d1_tile` as a package.

In [ ]:
import os
os.makedirs('/kaggle/working/src/d1_tile', exist_ok=True)
print(os.listdir('/kaggle/working/src'))


In [ ]:
%%writefile /kaggle/working/src/_blocks.py
"""Shared modern building blocks for d1 and d3.

All blocks use GroupNorm (not BatchNorm) so eval-mode and train-mode behave
identically at any batch size. INT8-friendly choices throughout.
"""
from __future__ import annotations

import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


def _safe_gn(ch: int, max_groups: int = 8) -> nn.GroupNorm:
    """Pick the largest divisor of ch that is <= max_groups (and >= 1)."""
    g = max_groups
    while g > 1 and ch % g != 0:
        g -= 1
    return nn.GroupNorm(g, ch)


# ----------------------------------------------------------------------- #
# GhostConv (Han et al., CVPR 2020) — half "primary" + half cheap depthwise
# ----------------------------------------------------------------------- #
class GhostConv(nn.Module):
    """Output channels = primary (out//2) ⊕ cheap depthwise of primary.

    Roughly halves the conv parameters vs a vanilla 3×3 conv at similar
    accuracy. Activations split out cleanly for INT8.
    """

    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1,
                 ratio: int = 2, gn_groups: int = 8):
        super().__init__()
        primary = out_ch // ratio
        cheap = out_ch - primary
        self.primary = nn.Conv2d(in_ch, primary, k, stride=s, padding=k // 2, bias=False)
        self.norm1 = _safe_gn(primary, gn_groups)
        self.cheap = nn.Conv2d(primary, cheap, 3, padding=1, groups=primary, bias=False)
        self.norm2 = _safe_gn(cheap, gn_groups)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        p = self.act(self.norm1(self.primary(x)))
        c = self.act(self.norm2(self.cheap(p)))
        return torch.cat([p, c], dim=1)


# ----------------------------------------------------------------------- #
# Squeeze-and-Excitation (Hu et al., CVPR 2018) — channel attention
# ----------------------------------------------------------------------- #
class SqueezeExcitation(nn.Module):
    def __init__(self, ch: int, r: int = 8):
        super().__init__()
        hidden = max(ch // r, 8)
        self.fc1 = nn.Conv2d(ch, hidden, 1)
        self.fc2 = nn.Conv2d(hidden, ch, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = F.adaptive_avg_pool2d(x, 1)
        s = F.silu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s))
        return x * s


# ----------------------------------------------------------------------- #
# RepVGG block (Ding et al., CVPR 2021) — train multi-branch / deploy 3x3
# ----------------------------------------------------------------------- #
class RepVGGBlock(nn.Module):
    """Three parallel branches at training time: 3×3 conv, 1×1 conv,
    identity. They sum into one output. At deployment time the three are
    fused into a single 3×3 conv (free accuracy/latency win).

    For our validation rounds we keep the train-time multi-branch path
    (no fusion). The fusion routine is left as a TODO for deployment.
    """

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, gn_groups: int = 8):
        super().__init__()
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.stride = stride
        self.conv3 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 1, stride=stride, padding=0, bias=False)
        self.norm3 = _safe_gn(out_ch, gn_groups)
        self.norm1 = _safe_gn(out_ch, gn_groups)
        self.has_identity = stride == 1 and in_ch == out_ch
        if self.has_identity:
            self.norm_id = _safe_gn(out_ch, gn_groups)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.norm3(self.conv3(x)) + self.norm1(self.conv1(x))
        if self.has_identity:
            y = y + self.norm_id(x)
        return self.act(y)


# ----------------------------------------------------------------------- #
# Lightweight neighborhood (linear) attention for tile propagation
# ----------------------------------------------------------------------- #
class NeighborhoodAttention2d(nn.Module):
    """Linear-attention version of HITNet's 3×3 tile-propagation step.

    Instead of a fixed 3×3 average over the 8 neighbours, each cell attends
    to its k×k neighbourhood with learned scores derived from a feature
    projection. Linear-attention kernel φ(x) = elu(x) + 1 keeps it cheap.
    """

    def __init__(self, ch: int, kernel: int = 3, head_dim: int = 16):
        super().__init__()
        self.kernel = kernel
        self.proj_qkv = nn.Conv2d(ch, 3 * head_dim, 1, bias=False)
        self.out = nn.Conv2d(head_dim, ch, 1, bias=False)
        self.head_dim = head_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        qkv = self.proj_qkv(x)                                          # (B, 3D, H, W)
        q, k, v = qkv.chunk(3, dim=1)
        q = F.elu(q) + 1
        k = F.elu(k) + 1

        # Unfold k and v over a kernel×kernel window for local linear attn.
        kh = self.kernel
        pad = kh // 2
        k_loc = F.unfold(k, kernel_size=kh, padding=pad)                # (B, D*K, H*W)
        v_loc = F.unfold(v, kernel_size=kh, padding=pad)
        k_loc = k_loc.view(B, self.head_dim, kh * kh, H * W)
        v_loc = v_loc.view(B, self.head_dim, kh * kh, H * W)
        q_flat = q.view(B, self.head_dim, 1, H * W)

        scores = (q_flat * k_loc).sum(dim=1, keepdim=True)              # (B,1,K,HW)
        weights = scores / (scores.sum(dim=2, keepdim=True) + 1e-6)
        out = (weights * v_loc).sum(dim=2)                              # (B, D, HW)
        out = out.view(B, self.head_dim, H, W)
        return self.out(out)


# ----------------------------------------------------------------------- #
# Pure-PyTorch selective-scan (Mamba S6 core), 1D over a chosen dim
# ----------------------------------------------------------------------- #
class SelectiveScan1d(nn.Module):
    """A simplified Mamba-S6 selective scan over one spatial dimension.

    Input  : x of shape (B, C, L) where L is the scan length.
    Output : same shape; aggregated context with input-dependent gating.

    For our stereo SGM-replacement use, the caller reshapes a cost volume
    (B, D, H, W) to (B*H, D, W) and back to scan along W (left-right), and
    similarly for the other three directions. Compute is sequential (slow
    on GPU, just like Python-loop SGM) but it is end-to-end differentiable
    and the operator is structurally identical to the official Mamba scan.
    """

    def __init__(self, ch: int, state_dim: int = 16, dt_rank: int = 4):
        super().__init__()
        self.ch = ch
        self.state_dim = state_dim
        self.dt_rank = dt_rank
        self.x_proj = nn.Conv1d(ch, dt_rank + 2 * state_dim, 1, bias=False)
        self.dt_proj = nn.Conv1d(dt_rank, ch, 1, bias=True)
        # Stable A-init: log(-A) where A < 0 for stability.
        A = torch.arange(1, state_dim + 1, dtype=torch.float32).repeat(ch, 1)  # (C, N)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(ch))
        self.gate = nn.Conv1d(ch, ch, 1, bias=True)

    def forward(self, x: torch.Tensor, reverse: bool = False,
                chunk_size: int = 16) -> torch.Tensor:
        """Chunked-parallel selective scan for h(t) = dA(t)·h(t-1) + dBu(t).

        Within each chunk we do a short sequential scan (chunk_size ≪ L),
        parallel across all chunks. Across chunks we propagate state with
        n_chunks sequential ops. Total CUDA launches ≈ chunk_size + n_chunks
        instead of L per direction. Numerically stable (no exp of cumulative
        log) for the sequence lengths we care about.
        """
        B, C, L = x.shape
        if reverse:
            x = x.flip(-1)

        proj = self.x_proj(x)                                            # (B, dt_rank + 2N, L)
        dt, B_t, C_t = torch.split(proj, [self.dt_rank, self.state_dim, self.state_dim], dim=1)
        dt = F.softplus(self.dt_proj(dt))                                # (B, C, L)

        A = -torch.exp(self.A_log).to(x.dtype)                           # (C, N), negative

        B_t = B_t.permute(0, 2, 1).unsqueeze(1)                          # (B, 1, L, N)
        C_t = C_t.permute(0, 2, 1).unsqueeze(1)                          # (B, 1, L, N)

        deltaA = torch.exp(dt.unsqueeze(-1) * A.view(1, C, 1, self.state_dim))   # (B, C, L, N)
        deltaB_u = dt.unsqueeze(-1) * B_t * x.unsqueeze(-1)              # (B, C, L, N)

        # Pad L to a multiple of chunk_size.
        pad = (chunk_size - L % chunk_size) % chunk_size
        if pad:
            deltaA = F.pad(deltaA, (0, 0, 0, pad), value=1.0)
            deltaB_u = F.pad(deltaB_u, (0, 0, 0, pad), value=0.0)
        Lp = L + pad
        n_chunks = Lp // chunk_size

        dA = deltaA.view(B, C, n_chunks, chunk_size, self.state_dim)
        dBu = deltaB_u.view(B, C, n_chunks, chunk_size, self.state_dim)

        # 1) Within each chunk: short sequential scan starting from h=0.
        h = torch.zeros(B, C, n_chunks, self.state_dim, device=x.device, dtype=x.dtype)
        within = []
        for t in range(chunk_size):
            h = dA[:, :, :, t] * h + dBu[:, :, :, t]
            within.append(h)
        within = torch.stack(within, dim=3)                              # (B, C, n_chunks, chunk_size, N)
        chunk_total_A = torch.prod(dA, dim=3)                            # (B, C, n_chunks, N)
        chunk_total_b = within[:, :, :, -1]                              # (B, C, n_chunks, N)

        # 2) Across chunks: sequential propagation of chunk-start state.
        h_start = torch.zeros(B, C, self.state_dim, device=x.device, dtype=x.dtype)
        starts = [h_start]
        for i in range(n_chunks - 1):
            h_start = chunk_total_A[:, :, i] * h_start + chunk_total_b[:, :, i]
            starts.append(h_start)
        starts = torch.stack(starts, dim=2)                              # (B, C, n_chunks, N)

        # 3) Combine: full h(t) = cumprod(dA[:t]) * h_start + within(t)
        cum_dA = torch.cumprod(dA, dim=3)                                # (B, C, n_chunks, chunk_size, N)
        h_full = cum_dA * starts.unsqueeze(3) + within                   # same shape
        h_full = h_full.view(B, C, Lp, self.state_dim)[:, :, :L]

        y = (h_full * C_t).sum(dim=-1)                                   # (B, C, L)
        y = y + x * self.D.view(1, C, 1)
        y = y * torch.sigmoid(self.gate(x))

        if reverse:
            y = y.flip(-1)
        return y


def scan_4_dirs(scanner: SelectiveScan1d, cv: torch.Tensor) -> torch.Tensor:
    """Apply a SelectiveScan1d in four directions over a (B, D, H, W) cost
    volume and sum the four scans. Sequential: slow but functionally
    identical to a CUDA Mamba kernel.
    """
    B, D, H, W = cv.shape
    # L→R / R→L scans operate over W (treat each row as an independent batch)
    cv_w = cv.permute(0, 2, 1, 3).reshape(B * H, D, W)
    lr = scanner(cv_w, reverse=False).view(B, H, D, W).permute(0, 2, 1, 3)
    rl = scanner(cv_w, reverse=True).view(B, H, D, W).permute(0, 2, 1, 3)
    # T→B / B→T over H
    cv_h = cv.permute(0, 3, 1, 2).reshape(B * W, D, H)
    tb = scanner(cv_h, reverse=False).view(B, W, D, H).permute(0, 2, 3, 1)
    bt = scanner(cv_h, reverse=True).view(B, W, D, H).permute(0, 2, 3, 1)
    return lr, rl, tb, bt


In [ ]:
%%writefile /kaggle/working/src/d1_tile/__init__.py
"""Kaggle build of StereoLite v8."""
from .model import StereoLite, StereoLiteConfig


In [ ]:
%%writefile /kaggle/working/src/d1_tile/tile_propagate.py
"""HITNet-inspired plane-tile hypothesis propagation for StereoLite v7.

Each tile carries a plane hypothesis:
    d    : disparity at the tile centre
    sx   : slope in x direction (sub-pixel disparity change per pixel to right)
    sy   : slope in y direction
    feat : learned per-tile features (propagated through iterations)
    conf : scalar confidence in [0, 1]

Key operations:
    - TileInit:     build a tile state from a local cost volume at coarse scale
    - TileRefine:   update (d, sx, sy, feat, conf) via warp-regress given L, R
                    features at the current scale. Warp uses the plane
                    equation, so matching benefits from sub-pixel gradients.
    - TileUpsample: apply the plane equation to produce 2x denser tiles at
                    the next scale up.

Plane equation for upsampling:
    for a parent tile centre at (y, x) with slope (sx, sy) and disparity d,
    its four children at offsets (dy, dx) in { -0.25, 0.25 } × { -0.25, 0.25 }
    get disparity d + sx * dx * parent_scale + sy * dy * parent_scale.
    (We implement this via explicit per-child offsets during upsample.)
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TileState:
    """Plane-tile hypothesis at some scale. All tensors are (B, C, H, W)
    where (H, W) is the tile-grid resolution."""
    d: torch.Tensor              # (B, 1, H, W) absolute disparity in 1/k-px units
    sx: torch.Tensor             # (B, 1, H, W) slope x (dispar change per pixel)
    sy: torch.Tensor             # (B, 1, H, W) slope y
    feat: torch.Tensor           # (B, Cf, H, W) learned per-tile features
    conf: torch.Tensor           # (B, 1, H, W) in [0, 1]


def _horizontal_warp(fR: torch.Tensor, disp: torch.Tensor) -> torch.Tensor:
    """Same warp primitive as the rest of the code."""
    B, _, H, W = fR.shape
    yy, xx = torch.meshgrid(
        torch.arange(H, device=fR.device, dtype=fR.dtype),
        torch.arange(W, device=fR.device, dtype=fR.dtype),
        indexing="ij",
    )
    gx = (xx.view(1, H, W) - disp.squeeze(1)) / max(W - 1, 1) * 2 - 1
    gy = (yy.view(1, H, W) / max(H - 1, 1) * 2 - 1).expand_as(gx)
    grid = torch.stack([gx, gy], dim=-1)
    return F.grid_sample(fR, grid, align_corners=True, padding_mode="border")


def _plane_warp_disp(tile: TileState) -> torch.Tensor:
    """For warping fR, use the plane-sampled disparity: each pixel's
    effective disparity includes a sub-pixel correction from the slope.
    Here we just return d — the slope-corrected sampling happens inside
    refine via slope-informed feature fusion. Keeping warp simple.
    """
    return tile.d


def _safe_gn(ch: int, max_groups: int = 8) -> nn.GroupNorm:
    g = 1
    for cand in range(min(max_groups, ch), 0, -1):
        if ch % cand == 0:
            g = cand
            break
    return nn.GroupNorm(num_groups=g, num_channels=ch)


class TileInit(nn.Module):
    """Initialise tiles at the coarsest scale (1/16) using a small local
    cost volume + 3D soft-argmin. Outputs (d, sx=0, sy=0, feat, conf)."""

    def __init__(self, feat_ch: int, max_disp: int = 24, groups: int = 8,
                 feat_out: int = 16):
        super().__init__()
        assert feat_ch % groups == 0
        self.max_disp = max_disp
        self.groups = groups
        self.feat_out = feat_out

        # Tiny 3D aggregator (~50 k params)
        self.agg = nn.Sequential(
            nn.Conv3d(groups, 16, 3, padding=1, bias=False),
            _safe_gn(16), nn.SiLU(inplace=True),
            nn.Conv3d(16, 16, 3, padding=1, bias=False),
            _safe_gn(16), nn.SiLU(inplace=True),
            nn.Conv3d(16, 1, 3, padding=1, bias=True),
        )
        # Per-tile feature extractor from left features only.
        self.feat_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_out, 3, padding=1, bias=False),
            _safe_gn(feat_out), nn.SiLU(inplace=True),
        )
        self.register_buffer(
            "disp_idx",
            torch.arange(max_disp, dtype=torch.float32).view(1, max_disp, 1, 1),
            persistent=False,
        )

    def forward(self, fL: torch.Tensor, fR: torch.Tensor) -> TileState:
        B, C, H, W = fL.shape
        cg = C // self.groups
        fL_g = fL.view(B, self.groups, cg, H, W)
        cv = fL.new_zeros((B, self.groups, self.max_disp, H, W))
        for d in range(self.max_disp):
            if d == 0:
                fR_s = fR
            else:
                fR_s = fL.new_zeros(fR.shape)
                fR_s[:, :, :, d:] = fR[:, :, :, :-d]
            fR_g = fR_s.view(B, self.groups, cg, H, W)
            cv[:, :, d] = (fL_g * fR_g).mean(dim=2)

        logits = self.agg(cv).squeeze(1)                     # (B, D, H, W)
        prob = F.softmax(logits, dim=1)
        d = (prob * self.disp_idx).sum(dim=1, keepdim=True)
        # Confidence from the peak mass — how sharp the distribution is
        conf = prob.max(dim=1, keepdim=True).values          # (B, 1, H, W)
        sx = torch.zeros_like(d)
        sy = torch.zeros_like(d)
        feat = self.feat_head(fL)
        return TileState(d=d, sx=sx, sy=sy, feat=feat, conf=conf)


class TileRefine(nn.Module):
    """Update a tile state given left/right CNN features at the tile scale.
    Warps fR using d, concatenates with fL, prior feat, slopes, and conf,
    then predicts residuals for d, sx, sy, feat, conf. Residuals are added.
    """

    def __init__(self, feat_ch: int, tile_feat_ch: int, hidden: int = 48):
        super().__init__()
        in_ch = 2 * feat_ch + tile_feat_ch + 4   # fL, fR_warp, feat, d, sx, sy, conf
        self.trunk = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1, bias=False),
            _safe_gn(hidden), nn.SiLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
            _safe_gn(hidden), nn.SiLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
            _safe_gn(hidden), nn.SiLU(inplace=True),
        )
        self.head_d = nn.Conv2d(hidden, 1, 1)
        self.head_sx = nn.Conv2d(hidden, 1, 1)
        self.head_sy = nn.Conv2d(hidden, 1, 1)
        self.head_conf = nn.Conv2d(hidden, 1, 1)
        self.head_feat = nn.Conv2d(hidden, tile_feat_ch, 1)

    def forward(self, tile: TileState, fL: torch.Tensor,
                fR: torch.Tensor) -> TileState:
        fR_w = _horizontal_warp(fR, _plane_warp_disp(tile))
        x = torch.cat([fL, fR_w, tile.feat, tile.d, tile.sx, tile.sy,
                        tile.conf], dim=1)
        h = self.trunk(x)
        d = F.softplus(self.head_d(h) + tile.d)              # keep positive
        sx = tile.sx + self.head_sx(h) * 0.1                 # small residual
        sy = tile.sy + self.head_sy(h) * 0.1
        conf = torch.sigmoid(self.head_conf(h) + 2.0 * tile.conf - 1.0)
        feat = tile.feat + self.head_feat(h)
        return TileState(d=d, sx=sx, sy=sy, feat=feat, conf=conf)


class TileUpsample(nn.Module):
    """Plane-aware upsample: the four children at (±0.25 tile) inherit
    d + slope * offset. We emit the 4 children tiles into a 2x denser grid.

    Slopes and features are just 2x bilinear upsampled (they change less
    rapidly than d across a scale boundary). Confidence is also bilinear.
    The disparity uses the plane equation for sub-pixel accuracy.
    """

    def __init__(self, scale_factor: int = 2):
        super().__init__()
        self.scale = scale_factor

    def forward(self, tile: TileState,
                target_hw: Optional[tuple[int, int]] = None) -> TileState:
        s = self.scale
        if target_hw is None:
            target_hw = (tile.d.shape[-2] * s, tile.d.shape[-1] * s)

        # Bilinear upsample of slopes, features, confidence
        sx_up = F.interpolate(tile.sx, size=target_hw, mode="bilinear",
                              align_corners=False)
        sy_up = F.interpolate(tile.sy, size=target_hw, mode="bilinear",
                              align_corners=False)
        feat_up = F.interpolate(tile.feat, size=target_hw, mode="bilinear",
                                 align_corners=False)
        conf_up = F.interpolate(tile.conf, size=target_hw, mode="bilinear",
                                 align_corners=False)

        # Plane-equation upsample of d: first bilinear the parent d (*s to
        # rescale the disparity units to the finer scale), then add a
        # sub-pixel correction from slopes.
        d_bilinear = F.interpolate(tile.d, size=target_hw, mode="bilinear",
                                    align_corners=False) * s

        # Build per-pixel offsets in the fine grid: each fine pixel sits
        # at a sub-parent-tile position. Offset in parent-tile units is in
        # {-0.25, +0.25} for a 2x upsample. We synthesise this pattern.
        device = tile.d.device
        H_f, W_f = target_hw
        # Offsets within each 2x2 block, in tile-local units (center-relative)
        dx_block = torch.tensor([[-0.25, 0.25], [-0.25, 0.25]],
                                device=device, dtype=tile.d.dtype)
        dy_block = torch.tensor([[-0.25, -0.25], [0.25, 0.25]],
                                device=device, dtype=tile.d.dtype)
        dx = dx_block.repeat(H_f // 2, W_f // 2)
        dy = dy_block.repeat(H_f // 2, W_f // 2)
        if dx.shape != (H_f, W_f):
            # Handle odd dims by pad/truncate (rare)
            dx = F.pad(dx, (0, W_f - dx.shape[1], 0, H_f - dx.shape[0]))
            dy = F.pad(dy, (0, W_f - dy.shape[1], 0, H_f - dy.shape[0]))
        dx = dx.view(1, 1, H_f, W_f)
        dy = dy.view(1, 1, H_f, W_f)

        # Slopes are in coarse units (disparity change per pixel at coarse
        # scale). Multiply by s to convert to fine units, then by the fine
        # per-pixel offset.
        d_plane = d_bilinear + (sx_up * s) * dx + (sy_up * s) * dy

        return TileState(d=d_plane, sx=sx_up, sy=sy_up, feat=feat_up,
                          conf=conf_up)


In [ ]:
%%writefile /kaggle/working/src/d1_tile/model.py
"""StereoLite v7 — tile-hypothesis propagation with iterative refinement.

Pipeline:
  Input (L, R) ──► GhostConv+SE encoder (shared) ──► f2, f4, f8, f16

  1/16: TileInit — tiny local cost volume + soft-argmin seeds
        (d, sx=0, sy=0, feat, conf). Iterative refine × N16.

  Upsample to 1/8 via plane equation (d + slope * offset) → iterate × N8.

  Upsample to 1/4 via plane equation → iterate × N4.

  Final: plane upsample 1/4 → 1/2 → full, with a single convex-mask-style
  learned upsample for sub-pixel boundary sharpness.

Key architectural changes vs v6:
  - Slopes (sx, sy) are now USED during upsampling (plane equation)
  - Iterative refinement: each scale runs the refine head N times
  - No 3D cost-volume aggregator at 1/8 (just local CV at 1/16 init)
  - No cascade CV (replaced by iteration)
  - DAv2 optional (kept for ablation/paper; default off)

Budget target: ~1.2-1.5 M trainable, ~150 ms inference on RTX 3050 at
512×832. Jetson INT8 target: ~25-40 ms.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from _blocks import (GhostConv, SqueezeExcitation, _safe_gn)

DAv2SmallFrozen = None  # not used on Kaggle (use_dav2=False only)
from .tile_propagate import (TileState, TileInit, TileRefine, TileUpsample)


def _gn(ch: int, groups: int = 8) -> nn.GroupNorm:
    return _safe_gn(ch, groups)


class GhostStage(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.down = GhostConv(in_ch, out_ch, k=3, s=stride)
        self.refine = GhostConv(out_ch, out_ch, k=3, s=1)
        self.se = SqueezeExcitation(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.se(self.refine(self.down(x)))


class TileFeatureEncoder(nn.Module):
    """Returns features at 1/2, 1/4, 1/8, 1/16."""

    def __init__(self, base: int = 24):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, base, 7, stride=2, padding=3, bias=False),
            _gn(base),
            nn.SiLU(inplace=True),
        )
        self.s4 = GhostStage(base, 2 * base, stride=2)
        self.s8 = GhostStage(2 * base, 3 * base, stride=2)
        self.s16 = GhostStage(3 * base, 4 * base, stride=2)
        self.out_channels = (base, 2 * base, 3 * base, 4 * base)

    def forward(self, x: torch.Tensor):
        x = x / 255.0
        f2 = self.stem(x)             # 1/2,  base ch
        f4 = self.s4(f2)              # 1/4,  2*base
        f8 = self.s8(f4)              # 1/8,  3*base
        f16 = self.s16(f8)            # 1/16, 4*base
        return f2, f4, f8, f16


class MobileNetV2Encoder(nn.Module):
    """ImageNet-pretrained MobileNetV2-100 features at 1/2, 1/4, 1/8, 1/16.

    Uses timm's features_only API and explicitly truncates blocks beyond the
    last requested stage. timm's FeatureListNet builds the full backbone and
    hooks intermediate outputs — the deeper unhooked blocks (1/32 stage)
    waste ~1 M params, consume forward compute, and break DDP because their
    parameters never receive gradients. Slicing self.backbone.blocks fixes
    all three problems.

    ImageNet normalisation is applied inside the module (input is expected
    in [0, 255] BGR→RGB-converted floats, to match the rest of the codebase).
    """

    def __init__(self, pretrained: bool = True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(
            "mobilenetv2_100", pretrained=pretrained,
            features_only=True, out_indices=(0, 1, 2, 3))
        # Drop unused 1/32 blocks. The deepest hook is at the end of the
        # last block whose feature stride <= 16. For mobilenetv2_100 this
        # is index 4 in self.backbone.blocks (0..6 total). Slicing keeps
        # blocks 0..4 inclusive.
        last_kept = self._deepest_used_block_idx()
        self.backbone.blocks = self.backbone.blocks[: last_kept + 1]
        ch = self.backbone.feature_info.channels()
        self.out_channels = tuple(ch)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1) * 255.0
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1) * 255.0
        self.register_buffer("mean", mean, persistent=False)
        self.register_buffer("std", std, persistent=False)

    def _deepest_used_block_idx(self) -> int:
        """Largest index in self.backbone.blocks whose output reduction is
        still <= 16 (our coarsest scale). Anything deeper is at 1/32 and we
        don't use it."""
        # feature_info.info is a list of dicts with 'reduction' and 'module'
        # ('module' looks like 'blocks.4', 'blocks.6', etc.).
        info = self.backbone.feature_info.info
        deepest = 0
        for entry in info:
            if entry.get("reduction", 0) <= 16:
                # Parse 'blocks.<N>' to get N
                module = entry.get("module", "")
                if module.startswith("blocks."):
                    deepest = max(deepest, int(module.split(".")[1]))
        return deepest

    def forward(self, x: torch.Tensor):
        x = (x - self.mean) / self.std
        f2, f4, f8, f16 = self.backbone(x)
        return f2, f4, f8, f16


class ConvexUpsample(nn.Module):
    """Learned 2x upsample of a (B, 1, H, W) disparity guided by per-pixel
    CNN features. Each fine pixel is a weighted sum of 9 coarse neighbours."""

    def __init__(self, feat_ch: int, scale: int = 2, hidden: int = 48):
        super().__init__()
        self.scale = scale
        self.mask = nn.Sequential(
            nn.Conv2d(feat_ch, hidden, 3, padding=1, bias=False),
            _gn(hidden), nn.SiLU(inplace=True),
            nn.Conv2d(hidden, 9 * scale * scale, 1),
        )

    def forward(self, disp: torch.Tensor, feat: torch.Tensor) -> torch.Tensor:
        B, _, H, W = disp.shape
        s = self.scale
        m = self.mask(feat).view(B, 1, 9, s, s, H, W).softmax(dim=2)
        up = F.unfold(disp * s, kernel_size=3, padding=1)
        up = up.view(B, 1, 9, 1, 1, H, W)
        out = (m * up).sum(dim=2)
        out = out.permute(0, 1, 4, 2, 5, 3).contiguous()
        return out.view(B, 1, s * H, s * W)


@dataclass
class StereoLiteConfig:
    base_ch: int = 24
    tile_feat_ch: int = 16         # channels carried in the tile feat state
    # Iterations at each scale:
    iters_16: int = 2
    iters_8: int = 3
    iters_4: int = 3
    # Cost volume at init
    init_max_disp: int = 24        # at 1/16, covers 24*16=384 full-res px
    init_groups: int = 8
    # Foundation model (off by default; keep code path for ablations)
    use_dav2: bool = False
    dav2_raw_proj_ch: int = 16
    # Refine head hidden channels
    refine_hidden: int = 48
    # Backbone: "ghost" (custom GhostConv) or "mobilenet" (ImageNet-pretrained)
    backbone: str = "ghost"
    backbone_pretrained: bool = True


class StereoLite(nn.Module):
    def __init__(self, cfg: StereoLiteConfig | None = None):
        super().__init__()
        self.cfg = cfg or StereoLiteConfig()
        b = self.cfg.base_ch
        if self.cfg.backbone == "mobilenet":
            self.fnet = MobileNetV2Encoder(pretrained=self.cfg.backbone_pretrained)
        else:
            self.fnet = TileFeatureEncoder(base=b)

        # Channel counts per scale from the encoder
        ch2, ch4, ch8, ch16 = self.fnet.out_channels

        # Optional DAv2 fusion only at 1/16 (keeps the path simple)
        if self.cfg.use_dav2:
            self.dav2 = DAv2SmallFrozen()
            self.dav2_proj16 = nn.Conv2d(
                self.dav2.embed_dim, self.cfg.dav2_raw_proj_ch, 1)
            ch16_fused = ch16 + self.cfg.dav2_raw_proj_ch
        else:
            self.dav2 = None
            ch16_fused = ch16

        # Groups need to divide feat_ch for the init cost volume
        g = self.cfg.init_groups
        while ch16_fused % g != 0 and g > 1:
            g -= 1
        self.init_tile = TileInit(feat_ch=ch16_fused,
                                   max_disp=self.cfg.init_max_disp,
                                   groups=g,
                                   feat_out=self.cfg.tile_feat_ch)

        # Refinement heads — one instance per scale (weights not shared
        # because feature channel counts differ).
        self.refine_16 = TileRefine(feat_ch=ch16_fused,
                                     tile_feat_ch=self.cfg.tile_feat_ch,
                                     hidden=self.cfg.refine_hidden)
        self.refine_8 = TileRefine(feat_ch=ch8,
                                    tile_feat_ch=self.cfg.tile_feat_ch,
                                    hidden=self.cfg.refine_hidden)
        self.refine_4 = TileRefine(feat_ch=ch4,
                                    tile_feat_ch=self.cfg.tile_feat_ch,
                                    hidden=self.cfg.refine_hidden)

        # Plane upsamples (no trainable weights, just geometry)
        self.up_16_to_8 = TileUpsample(scale_factor=2)
        self.up_8_to_4 = TileUpsample(scale_factor=2)

        # Final learned 4x upsample 1/4 → full, guided by 1/2 features
        # (done in two steps: 1/4 → 1/2 using f4 features, 1/2 → full
        # using f2). Two small convex modules.
        self.up_final_4_to_2 = ConvexUpsample(feat_ch=ch4, scale=2)
        self.up_final_2_to_1 = ConvexUpsample(feat_ch=ch2, scale=2)

    def _tile_d_at_scale(self, tile: TileState, scale: int) -> torch.Tensor:
        """Return the tile's d in 1/scale-px units (what the tile stores)."""
        return tile.d

    def forward(self, left: torch.Tensor, right: torch.Tensor,
                aux: bool = False):
        fL2, fL4, fL8, fL16 = self.fnet(left)
        fR2, fR4, fR8, fR16 = self.fnet(right)

        if self.dav2 is not None:
            B = left.shape[0]
            out = self.dav2(torch.cat([left, right], dim=0))
            raw16 = self.dav2_proj16(
                F.interpolate(out["raw"], size=fL16.shape[-2:],
                              mode="bilinear", align_corners=False))
            dL16, dR16 = raw16[:B], raw16[B:]
            fL16_ = torch.cat([fL16, dL16], dim=1)
            fR16_ = torch.cat([fR16, dR16], dim=1)
        else:
            fL16_, fR16_ = fL16, fR16

        # 1/16: init + iterate
        tile = self.init_tile(fL16_, fR16_)
        t16_stages = [tile]
        for _ in range(self.cfg.iters_16):
            tile = self.refine_16(tile, fL16_, fR16_)
            t16_stages.append(tile)

        # 1/16 → 1/8 via plane equation
        tile = self.up_16_to_8(tile, target_hw=fL8.shape[-2:])
        t8_stages = [tile]
        for _ in range(self.cfg.iters_8):
            tile = self.refine_8(tile, fL8, fR8)
            t8_stages.append(tile)

        # 1/8 → 1/4
        tile = self.up_8_to_4(tile, target_hw=fL4.shape[-2:])
        t4_stages = [tile]
        for _ in range(self.cfg.iters_4):
            tile = self.refine_4(tile, fL4, fR4)
            t4_stages.append(tile)

        # Final: learned upsample 1/4 → 1/2 → full, each 2x. Disparity
        # unit conversion: multiplying by 2 each time converts from
        # coarse-unit disparity to fine-unit.
        d_half = self.up_final_4_to_2(tile.d, fL4)
        d_full = self.up_final_2_to_1(d_half, fL2)
        if d_full.shape[-2:] != left.shape[-2:]:
            d_full = F.interpolate(d_full, size=left.shape[-2:],
                                    mode="bilinear", align_corners=True)

        if aux:
            return {
                "d_final": d_full,
                # expose a few iteration outputs for multi-scale loss
                "d_half": d_half,
                "d4": tile.d,               # after last 1/4 iteration
                "d8": t8_stages[-1].d,      # after last 1/8 iteration
                "d8_cv": t16_stages[-1].d,  # backward-compat key for loss
                "d16": t16_stages[-1].d,
                "d32": t16_stages[0].d,     # init at 1/16
            }
        return d_full




In [ ]:
%%writefile /kaggle/working/src/sceneflow_loader.py
"""Scene Flow Driving subset dataset loader.

After extracting:
    data/sceneflow_driving/frames_finalpass/{15,35}mm_focallength/scene_*/{left,right}/*.png
    data/sceneflow_driving/disparity/{15,35}mm_focallength/scene_*/left/*.pfm

This module exposes:
    enumerate_pairs(root)           -> list of (left_png, right_png, left_pfm)
    train_val_split(items, n_val)   -> (train, val) deterministic
    SceneFlowDataset                -> torch.utils.data.Dataset

PFM read code is borrowed from the standard Middlebury format; values are
positive disparities, sentinel `inf` marks invalid pixels.
"""
from __future__ import annotations

import os
import re
import struct
from pathlib import Path

import numpy as np
import torch
import cv2


# ---------------------------- PFM reader ---------------------------------- #
def read_pfm(path: str) -> np.ndarray:
    with open(path, "rb") as fp:
        header = fp.readline().decode("latin-1").rstrip()
        if header == "PF":
            color = True
        elif header == "Pf":
            color = False
        else:
            raise ValueError(f"Not a PFM file: {path}")
        line = fp.readline().decode("latin-1")
        m = re.match(r"^(\d+)\s+(\d+)\s*$", line)
        if not m:
            raise ValueError(f"Bad PFM header in {path}")
        w, h = int(m.group(1)), int(m.group(2))
        scale = float(fp.readline().decode("latin-1").rstrip())
        endian = "<" if scale < 0 else ">"
        data = np.frombuffer(fp.read(), dtype=endian + "f")
        if color:
            data = data.reshape(h, w, 3)
        else:
            data = data.reshape(h, w)
        data = np.flipud(data)        # PFM is bottom-to-top
        return data.copy()


# ---------------------------- pair enumeration ---------------------------- #
def enumerate_pairs(root: str) -> list[tuple[str, str, str]]:
    """Walk frames_finalpass and find matching disparity PFMs.

    Returns a list of (left_png, right_png, left_pfm).
    """
    rgb_root = os.path.join(root, "frames_finalpass")
    disp_root = os.path.join(root, "disparity")
    out: list[tuple[str, str, str]] = []
    for fl_dir in sorted(os.listdir(rgb_root)):
        # 15mm_focallength / 35mm_focallength
        for direction in sorted(os.listdir(os.path.join(rgb_root, fl_dir))):
            # scene_backwards / scene_forwards
            for speed in sorted(os.listdir(os.path.join(rgb_root, fl_dir, direction))):
                # fast / slow
                left_dir = os.path.join(rgb_root, fl_dir, direction, speed, "left")
                right_dir = os.path.join(rgb_root, fl_dir, direction, speed, "right")
                disp_dir = os.path.join(disp_root, fl_dir, direction, speed, "left")
                if not os.path.isdir(left_dir):
                    continue
                for fname in sorted(os.listdir(left_dir)):
                    if not fname.endswith(".png"):
                        continue
                    lp = os.path.join(left_dir, fname)
                    rp = os.path.join(right_dir, fname)
                    pp = os.path.join(disp_dir, fname.replace(".png", ".pfm"))
                    if os.path.exists(rp) and os.path.exists(pp):
                        out.append((lp, rp, pp))
    return out


def train_val_split(items: list, n_val: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(items))
    rng.shuffle(idx)
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]
    return [items[i] for i in train_idx], [items[i] for i in val_idx]


# ---------------------------- torch Dataset ------------------------------- #
class SceneFlowDriving(torch.utils.data.Dataset):
    """Returns (left_t, right_t, disp_t) at fixed (H, W). disp_t is in pixels
    at the saved (H, W); invalid pixels are 0."""

    def __init__(self, items: list, h: int, w: int):
        self.items = items
        self.h = h
        self.w = w

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        lp, rp, pp = self.items[idx]
        L = cv2.imread(lp)
        R = cv2.imread(rp)
        D_full = read_pfm(pp)        # (H, W) in pixels at native resolution
        H_native, W_native = D_full.shape
        L = cv2.resize(L, (self.w, self.h), interpolation=cv2.INTER_AREA)
        R = cv2.resize(R, (self.w, self.h), interpolation=cv2.INTER_AREA)
        # Resize disparity proportionally: a disparity of d_native at (H_native, W_native)
        # corresponds to d_native * (W/W_native) at (H, W).
        sx = self.w / W_native
        D = cv2.resize(D_full, (self.w, self.h), interpolation=cv2.INTER_LINEAR) * sx
        D[~np.isfinite(D) | (D < 0)] = 0
        Lt = torch.from_numpy(cv2.cvtColor(L, cv2.COLOR_BGR2RGB)).float().permute(2, 0, 1)
        Rt = torch.from_numpy(cv2.cvtColor(R, cv2.COLOR_BGR2RGB)).float().permute(2, 0, 1)
        Dt = torch.from_numpy(D.astype(np.float32)).unsqueeze(0)
        return Lt, Rt, Dt


if __name__ == "__main__":
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="/home/abrar/Research/stero_research_claude/data/sceneflow_driving")
    p.add_argument("--n", type=int, default=5)
    args = p.parse_args()
    items = enumerate_pairs(args.root)
    print(f"found {len(items)} stereo pairs under {args.root}")
    for lp, rp, pp in items[:args.n]:
        d = read_pfm(pp)
        print(f"  {os.path.basename(lp)}  disp range [{np.nanmin(d):.1f}, {np.nanmax(d):.1f}], shape={d.shape}")


In [ ]:
%%writefile /kaggle/working/train_ddp.py
"""DDP trainer for StereoLite v8 on Kaggle 2x T4 GPUs.

Launched via `torchrun --nproc_per_node=2 train_ddp.py ...`.

Highlights:
  - MobileNetV2-100 ImageNet-pretrained encoder
  - OneCycle LR peak 8e-4 (OpenStereo recipe)
  - Random-erase on right image
  - Multi-scale L1 + grad + bad-1 hinge + edge-aware smoothness loss
  - FP16 autocast + GradScaler
  - DistributedDataParallel on all visible GPUs
  - Rank-0 saves best-EPE checkpoint + CSV log each epoch
"""
from __future__ import annotations

import argparse
import csv
import math
import os
import sys
import time

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

# Source layout set up by the notebook: /kaggle/working/src/
sys.path.insert(0, "/kaggle/working/src")
os.environ.setdefault("XFORMERS_DISABLED", "1")

from d1_tile import StereoLite, StereoLiteConfig
from sceneflow_loader import enumerate_pairs, train_val_split, SceneFlowDriving, read_pfm


# ---------------------------- loss terms ---------------------------------- #
def sobel_xy(d):
    kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                      device=d.device, dtype=d.dtype).view(1, 1, 3, 3) / 8
    ky = kx.transpose(-1, -2).contiguous()
    d_pad = F.pad(d, (1, 1, 1, 1), mode="replicate")
    return F.conv2d(d_pad, kx), F.conv2d(d_pad, ky)


def grad_loss(pred, target, valid):
    gx_p, gy_p = sobel_xy(pred)
    gx_t, gy_t = sobel_xy(target)
    err = ((gx_p - gx_t).abs() + (gy_p - gy_t).abs()) * valid
    return err.sum() / valid.sum().clamp(min=1.0)


def edge_aware_smooth(pred, image, sigma=3.0):
    dx_p = (pred[..., 1:] - pred[..., :-1]).abs()
    dy_p = (pred[..., 1:, :] - pred[..., :-1, :]).abs()
    g = image.mean(dim=1, keepdim=True)
    dx_i = (g[..., 1:] - g[..., :-1]).abs()
    dy_i = (g[..., 1:, :] - g[..., :-1, :]).abs()
    wx = torch.exp(-dx_i / sigma)
    wy = torch.exp(-dy_i / sigma)
    return (wx * dx_p).mean() + (wy * dy_p).mean()


SCALE_WEIGHTS = {
    "d_final": 1.0, "d_half": 0.7, "d4": 0.5,
    "d8": 0.3, "d8_cv": 1.0, "d16": 0.3, "d32": 0.2,
}


def multiscale_loss(preds, D_full, grad_w=0.5, hinge_w=0.3,
                     smooth_w=0.02, left_img=None, max_disp=192.0):
    total = torch.zeros((), device=D_full.device, dtype=D_full.dtype)
    diag = {}
    for key, pred in preds.items():
        w = SCALE_WEIGHTS.get(key)
        if w is None:
            continue
        th = pred.shape[-2:]
        scale = th[1] / D_full.shape[-1]
        D_s = F.interpolate(D_full, size=th, mode="bilinear",
                             align_corners=True) * scale
        D_s[~torch.isfinite(D_s)] = 0
        valid = (D_s > 0) & (D_s < max_disp * scale) & torch.isfinite(D_s)
        n = valid.sum().clamp(min=1.0)
        diff = (pred - D_s).abs() * valid
        l1 = diff.sum() / n
        g = grad_loss(pred, D_s, valid) if w > 0.1 else torch.zeros_like(l1)
        thresh = 1.0 * scale
        hinge = (torch.relu(diff - thresh)).sum() / n
        total = total + w * (l1 + grad_w * g + hinge_w * hinge)
        diag[key] = float(l1.item())
    if smooth_w > 0 and left_img is not None and "d_final" in preds:
        total = total + smooth_w * edge_aware_smooth(preds["d_final"], left_img)
    return total, diag


def random_erase_right(R, prob=0.3, n_patches=2, h_frac=(0.02, 0.1),
                        w_frac=(0.02, 0.1)):
    if prob <= 0:
        return R
    B, _, H, W = R.shape
    R = R.clone()
    for b in range(B):
        if torch.rand(1).item() > prob:
            continue
        for _ in range(n_patches):
            ph = int(H * (h_frac[0] + torch.rand(1).item() * (h_frac[1] - h_frac[0])))
            pw = int(W * (w_frac[0] + torch.rand(1).item() * (w_frac[1] - w_frac[0])))
            if ph < 1 or pw < 1:
                continue
            y0 = torch.randint(0, max(H - ph, 1), (1,)).item()
            x0 = torch.randint(0, max(W - pw, 1), (1,)).item()
            fill = R[b, :, y0:y0 + ph, x0:x0 + pw].mean(dim=(-1, -2), keepdim=True)
            R[b, :, y0:y0 + ph, x0:x0 + pw] = fill
    return R


# ---------------------------- DDP helpers --------------------------------- #
def is_dist():
    return dist.is_available() and dist.is_initialized()


def get_rank():
    return dist.get_rank() if is_dist() else 0


def get_world_size():
    return dist.get_world_size() if is_dist() else 1


def setup_dist():
    if "RANK" in os.environ:
        dist.init_process_group(backend="nccl")
        lr = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(lr)
        return lr
    return 0


# ---------------------------- validation --------------------------------- #
@torch.no_grad()
def validate(model, val_loader, device, max_disp=192.0):
    model.eval()
    epes, bad1s = [], []
    for L, R, D in val_loader:
        L, R, D = L.to(device), R.to(device), D.to(device)
        pred = model(L, R)
        if pred.shape != D.shape:
            pred = F.interpolate(pred, size=D.shape[-2:], mode="bilinear",
                                  align_corners=True)
        valid = (D > 0) & (D < max_disp) & torch.isfinite(D)
        n = valid.sum().clamp(min=1.0)
        epes.append(float(((pred - D).abs() * valid).sum() / n))
        bad1s.append(float((((pred - D).abs() > 1.0).float() * valid).sum() / n))
    model.train()
    return float(np.mean(epes)), float(np.mean(bad1s))


def _colourise(d, lo, hi):
    d = d.astype(np.float32)
    v = np.clip((d - lo) / max(hi - lo, 1e-6), 0, 1) * 255
    return cv2.applyColorMap(v.astype(np.uint8), cv2.COLORMAP_TURBO)


def _annotate(img, text, h=22):
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], h), (0, 0, 0), -1)
    cv2.putText(out, text, (6, h - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.42,
                (255, 255, 255), 1, cv2.LINE_AA)
    return out


@torch.no_grad()
def save_sample_panels(model, track_pairs, device, out_dir, tag,
                        inf_h, inf_w):
    """Save per-step montage showing [L | GT | pred] for each tracked pair.

    Writes to `<out_dir>/samples/step_<tag>.png`. `tag` is typically
    a zero-padded global step, e.g. "00500".
    """
    os.makedirs(os.path.join(out_dir, "samples"), exist_ok=True)
    was_training = model.training
    model.eval()
    rows = []
    ms_list = []
    for i, (lp, rp, pp) in enumerate(track_pairs):
        L = cv2.imread(lp); R = cv2.imread(rp)
        D_n = read_pfm(pp)
        H_n, W_n = D_n.shape
        L_in = cv2.resize(L, (inf_w, inf_h))
        R_in = cv2.resize(R, (inf_w, inf_h))
        sx = inf_w / W_n
        D = cv2.resize(D_n, (inf_w, inf_h)) * sx
        D[~np.isfinite(D) | (D < 0)] = 0

        Lt = torch.from_numpy(cv2.cvtColor(L_in, cv2.COLOR_BGR2RGB)).float() \
                .permute(2, 0, 1).unsqueeze(0).to(device)
        Rt = torch.from_numpy(cv2.cvtColor(R_in, cv2.COLOR_BGR2RGB)).float() \
                .permute(2, 0, 1).unsqueeze(0).to(device)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        P = model(Lt, Rt).squeeze().float().cpu().numpy()
        if device.type == "cuda":
            torch.cuda.synchronize()
        ms_list.append((time.time() - t0) * 1000)

        valid = (D > 0) & (D < 192) & np.isfinite(D)
        if valid.sum() > 16:
            lo = float(np.percentile(D[valid], 2))
            hi = float(np.percentile(D[valid], 98))
        else:
            lo, hi = 0.0, 60.0
        epe = float((np.abs(P - D) * valid).sum() / max(valid.sum(), 1))

        row = np.hstack([
            _annotate(L_in, f"L pair{i:02d}  {inf_w}x{inf_h}"),
            _annotate(_colourise(D, lo, hi), f"GT"),
            _annotate(_colourise(P, lo, hi),
                      f"pred  EPE={epe:.2f}  {ms_list[-1]:.0f}ms  step {tag}"),
        ])
        rows.append(row)

    montage = np.vstack(rows)
    out_path = os.path.join(out_dir, "samples", f"step_{tag}.png")
    cv2.imwrite(out_path, montage)
    if was_training:
        model.train()
    return float(np.mean(ms_list))


# ---------------------------- main --------------------------------------- #
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data_root", required=True)
    p.add_argument("--out_dir", default="/kaggle/working")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch", type=int, default=4)
    p.add_argument("--inf_h", type=int, default=384)
    p.add_argument("--inf_w", type=int, default=768)
    p.add_argument("--n_val", type=int, default=200)
    p.add_argument("--max_train", type=int, default=0,
                   help="cap training set (0 = all)")
    p.add_argument("--lr", type=float, default=8e-4)
    p.add_argument("--random_erase_p", type=float, default=0.3)
    p.add_argument("--grad_w", type=float, default=0.5)
    p.add_argument("--hinge_w", type=float, default=0.3)
    p.add_argument("--smooth_w", type=float, default=0.02)
    p.add_argument("--num_workers", type=int, default=2)
    p.add_argument("--log_every", type=int, default=50)
    p.add_argument("--sample_every", type=int, default=500,
                   help="save iterative sample-panel montage every N steps (0 = off)")
    p.add_argument("--n_sample_pairs", type=int, default=6,
                   help="number of val pairs to render in each sample montage")
    p.add_argument("--resume", default="")
    args = p.parse_args()

    local_rank = setup_dist()
    device = torch.device(f"cuda:{local_rank}")
    is_main = get_rank() == 0
    ws = get_world_size()

    if is_main:
        os.makedirs(args.out_dir, exist_ok=True)
        print(f"world size={ws}  local_rank={local_rank}  device={device}")
        print(f"data_root={args.data_root}  out_dir={args.out_dir}")

    # ---- data ----
    items = enumerate_pairs(args.data_root)
    train_items, val_items = train_val_split(items, args.n_val)
    if args.max_train > 0:
        train_items = train_items[: args.max_train]
    # Evenly-spaced val subset used for the sample-panel montage
    track_pairs = []
    if args.sample_every > 0 and len(val_items) > 0:
        stride = max(1, len(val_items) // args.n_sample_pairs)
        track_pairs = [val_items[i * stride] for i in range(args.n_sample_pairs)
                        if i * stride < len(val_items)]
    if is_main:
        print(f"train={len(train_items)}  val={len(val_items)}  "
              f"sample_track={len(track_pairs)}")

    train_ds = SceneFlowDriving(train_items, args.inf_h, args.inf_w)
    val_ds = SceneFlowDriving(val_items, args.inf_h, args.inf_w)

    train_sampler = (DistributedSampler(train_ds, num_replicas=ws,
                                         rank=get_rank(), shuffle=True)
                     if is_dist() else None)
    train_loader = DataLoader(train_ds, batch_size=args.batch,
                               sampler=train_sampler,
                               shuffle=(train_sampler is None),
                               num_workers=args.num_workers, pin_memory=True,
                               persistent_workers=args.num_workers > 0,
                               drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                             num_workers=1, pin_memory=True)

    # ---- model ----
    model = StereoLite(StereoLiteConfig(backbone="mobilenet",
                                     use_dav2=False)).to(device)
    if args.resume and os.path.exists(args.resume):
        ck = torch.load(args.resume, map_location="cpu", weights_only=False)
        sd = ck["model"] if "model" in ck else ck
        model.load_state_dict(sd, strict=False)
        if is_main:
            print(f"resumed from {args.resume}")

    if is_dist():
        model = DDP(model, device_ids=[local_rank],
                    find_unused_parameters=False)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if is_main:
        print(f"trainable params: {trainable/1e6:.3f} M")

    # ---- optim ----
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=args.lr, weight_decay=1e-5)
    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * args.epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=args.lr, total_steps=total_steps,
        pct_start=0.05, anneal_strategy="cos",
        div_factor=25.0, final_div_factor=1e4)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    # ---- log ----
    if is_main:
        log_path = os.path.join(args.out_dir, "train_log.csv")
        fp_log = open(log_path, "w", newline="")
        w_csv = csv.writer(fp_log)
        w_csv.writerow(["epoch", "step", "loss", "l1_final", "l1_cv",
                         "lr", "ms_per_step"])

    best_epe = float("inf")
    global_step = 0
    model.train()
    t_start = time.time()

    for epoch in range(1, args.epochs + 1):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        t_epoch = time.time()
        run_loss, run_l1f, run_l1c = [], [], []

        for i, (L, R, D) in enumerate(train_loader):
            L, R, D = L.to(device, non_blocking=True), \
                      R.to(device, non_blocking=True), \
                      D.to(device, non_blocking=True)
            R = random_erase_right(R, prob=args.random_erase_p)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                preds = model(L, R, aux=True)
                if preds["d_final"].shape[-2:] != D.shape[-2:]:
                    preds["d_final"] = F.interpolate(
                        preds["d_final"], size=D.shape[-2:],
                        mode="bilinear", align_corners=True)
                preds_fp32 = {k: v.float() for k, v in preds.items()}
                loss, diag = multiscale_loss(
                    preds_fp32, D.float(),
                    grad_w=args.grad_w, hinge_w=args.hinge_w,
                    smooth_w=args.smooth_w, left_img=L.float())

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()

            global_step += 1
            run_loss.append(loss.item())
            run_l1f.append(diag.get("d_final", 0.0))
            run_l1c.append(diag.get("d8_cv", 0.0))

            if is_main and (global_step == 1 or global_step % args.log_every == 0):
                ms = (time.time() - t_start) * 1000 / global_step
                cur_lr = opt.param_groups[0]["lr"]
                ml = float(np.mean(run_loss[-args.log_every:]))
                m_l1 = float(np.mean(run_l1f[-args.log_every:]))
                m_cv = float(np.mean(run_l1c[-args.log_every:]))
                w_csv.writerow([epoch, global_step, ml, m_l1, m_cv,
                                cur_lr, ms])
                fp_log.flush()
                print(f"ep {epoch:3d} step {global_step:6d}  "
                      f"loss={ml:6.3f}  l1_f={m_l1:6.3f}  l1_cv={m_cv:6.3f}  "
                      f"lr={cur_lr:.2e}  {ms:5.0f} ms/step")

            # Save iterative improvement panels (rank 0 only)
            if (is_main and track_pairs and args.sample_every > 0
                    and global_step % args.sample_every == 0):
                core = model.module if is_dist() else model
                med_ms = save_sample_panels(
                    core, track_pairs, device, args.out_dir,
                    tag=f"{global_step:06d}",
                    inf_h=args.inf_h, inf_w=args.inf_w)
                print(f"    sample panel saved  "
                      f"({len(track_pairs)} pairs, median {med_ms:.0f} ms)")

        # End-of-epoch validation on rank 0 only
        if is_main:
            epe, bad1 = validate(model.module if is_dist() else model,
                                  val_loader, device)
            dt = time.time() - t_epoch
            print(f"==> epoch {epoch}: VAL EPE={epe:.3f} px  "
                  f"bad1={bad1*100:.1f}%  ({dt/60:.1f} min)")

            ck = {"model": (model.module if is_dist() else model).state_dict(),
                  "epe": epe, "bad1": bad1, "epoch": epoch,
                  "args": vars(args)}
            torch.save(ck, os.path.join(args.out_dir, "stereolite_v8_last.pth"))
            if epe < best_epe:
                best_epe = epe
                torch.save(ck, os.path.join(args.out_dir, "stereolite_v8_best.pth"))
                print(f"    saved new best (EPE {epe:.3f})")

        if is_dist():
            dist.barrier()

    if is_main:
        fp_log.close()
        print(f"\ntraining done. best VAL EPE = {best_epe:.3f} px")
        print(f"total time: {(time.time()-t_start)/60:.1f} min")

    if is_dist():
        dist.destroy_process_group()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/export_v8.py
"""Export StereoLite v8 to multiple deployment formats.

Called from the notebook after training. Produces:
  - stereolite_v8.pth         (fp32 state_dict, already saved by trainer)
  - stereolite_v8.ts.pt       (TorchScript, trace-mode, fp32, CUDA)
  - stereolite_v8_fp32.onnx   (ONNX fp32, opset 17)
  - stereolite_v8_fp16.onnx   (ONNX fp16, via onnxconverter_common)
  - stereolite_v8_int8.onnx   (ONNX int8, dynamic quantization of conv/gemm)

Validates each export by running a random input through PyTorch and the
exported form, reporting max disparity diff.
"""
from __future__ import annotations

import argparse
import os
import sys
import time

import numpy as np
import torch

sys.path.insert(0, "/kaggle/working/src")
os.environ.setdefault("XFORMERS_DISABLED", "1")

from d1_tile import StereoLite, StereoLiteConfig


def load_model(ckpt_path: str, device: torch.device) -> torch.nn.Module:
    model = StereoLite(StereoLiteConfig(backbone="mobilenet",
                                     use_dav2=False)).to(device)
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = ck["model"] if "model" in ck else ck
    model.load_state_dict(sd, strict=True)
    # eval mode is required for clean ONNX export (folds BN running stats in
    # MobileNetV2). Our other norms are GroupNorm, train/eval-identical.
    model.eval()
    return model


def export_torchscript(model, out_path, inp):
    with torch.no_grad():
        ts = torch.jit.trace(model, inp, check_trace=False)
    ts.save(out_path)
    print(f"  -> {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)")


def export_onnx(model, out_path, inp, opset=17):
    # Use the legacy tracing exporter — the dynamo exporter currently mis-names
    # weight initializers when the model contains timm BatchNorms alongside
    # GroupNorms (produces an invalid graph onnxruntime refuses to load).
    torch.onnx.export(
        model, inp, out_path,
        input_names=["left", "right"],
        output_names=["disparity"],
        opset_version=opset,
        do_constant_folding=True,
        dynamo=False,
    )
    print(f"  -> {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)")


def convert_onnx_fp16(fp32_path, fp16_path):
    import onnx
    from onnxconverter_common import float16
    m = onnx.load(fp32_path)
    m16 = float16.convert_float_to_float16(m, keep_io_types=False)
    onnx.save(m16, fp16_path)
    print(f"  -> {fp16_path}  ({os.path.getsize(fp16_path)/1e6:.1f} MB)")


def quantize_onnx_int8(fp32_path, int8_path):
    from onnxruntime.quantization import quantize_dynamic, QuantType
    from onnxruntime.quantization.shape_inference import quant_pre_process
    # Preprocess: unfold constant-folded ops so every conv weight is a direct
    # initializer. Without this, quantize_dynamic errors on folded convs.
    pre_path = fp32_path.replace(".onnx", "_pre.onnx")
    quant_pre_process(fp32_path, pre_path, skip_symbolic_shape=False)
    quantize_dynamic(pre_path, int8_path, weight_type=QuantType.QInt8)
    os.remove(pre_path)
    print(f"  -> {int8_path}  ({os.path.getsize(int8_path)/1e6:.1f} MB)")


def compare_pytorch_vs_onnx(pytorch_model, onnx_path, inp, device, tol=1e-2):
    import onnxruntime as ort
    with torch.no_grad():
        out_torch = pytorch_model(*inp).cpu().numpy()
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
                if device.type == "cuda" else ["CPUExecutionProvider"]
    try:
        sess = ort.InferenceSession(onnx_path, providers=providers)
    except Exception as e:
        print(f"    onnxruntime load failed: {e}")
        return
    ort_inp = {
        "left": inp[0].cpu().numpy(),
        "right": inp[1].cpu().numpy(),
    }
    out_ort = sess.run(["disparity"], ort_inp)[0]
    diff = np.abs(out_torch - out_ort)
    print(f"    max abs diff vs PyTorch: {diff.max():.4f}  "
          f"mean: {diff.mean():.4f}")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", default="/kaggle/working/stereolite_v8_best.pth")
    p.add_argument("--out_dir", default="/kaggle/working")
    p.add_argument("--inf_h", type=int, default=384)
    p.add_argument("--inf_w", type=int, default=768)
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}  ckpt={args.ckpt}")
    model = load_model(args.ckpt, device)

    L = torch.rand(1, 3, args.inf_h, args.inf_w, device=device) * 255
    R = torch.rand(1, 3, args.inf_h, args.inf_w, device=device) * 255
    with torch.no_grad():
        _ = model(L, R)  # warmup

    ts_path = os.path.join(args.out_dir, "stereolite_v8.ts.pt")
    fp32_onnx = os.path.join(args.out_dir, "stereolite_v8_fp32.onnx")
    fp16_onnx = os.path.join(args.out_dir, "stereolite_v8_fp16.onnx")
    int8_onnx = os.path.join(args.out_dir, "stereolite_v8_int8.onnx")

    print("\n[1/5] TorchScript (CUDA, fp32)")
    export_torchscript(model, ts_path, (L, R))

    print("\n[2/5] ONNX fp32  (opset 17)")
    # ONNX export runs on CPU for reproducibility; cast inputs.
    model_cpu = model.cpu()
    L_cpu, R_cpu = L.cpu(), R.cpu()
    export_onnx(model_cpu, fp32_onnx, (L_cpu, R_cpu))

    print("\n[3/5] ONNX fp16")
    convert_onnx_fp16(fp32_onnx, fp16_onnx)

    print("\n[4/5] ONNX int8  (dynamic quantization)")
    quantize_onnx_int8(fp32_onnx, int8_onnx)

    print("\n[5/5] validation (PyTorch vs ONNX fp32)")
    compare_pytorch_vs_onnx(model_cpu, fp32_onnx, (L_cpu, R_cpu),
                             torch.device("cpu"))

    print("\ndone.")


if __name__ == "__main__":
    main()


## 5. Sanity check model assembles + forward pass works

In [ ]:
import sys, os
os.environ.setdefault('XFORMERS_DISABLED', '1')
sys.path.insert(0, '/kaggle/working/src')
import torch
from d1_tile import StereoLite, StereoLiteConfig

m = StereoLite(StereoLiteConfig(backbone='mobilenet', use_dav2=False)).cuda()
L = torch.rand(1, 3, INF_H, INF_W, device='cuda') * 255
R = torch.rand(1, 3, INF_H, INF_W, device='cuda') * 255
with torch.no_grad():
    out = m(L, R)
print(f'trainable params: {sum(p.numel() for p in m.parameters())/1e6:.3f} M')
print(f'output shape: {tuple(out.shape)}')
del m, out
torch.cuda.empty_cache()


## 6. Verify dataset is reachable

In [ ]:
sys.path.insert(0, '/kaggle/working/src')
from sceneflow_loader import enumerate_pairs
items = enumerate_pairs(DATA_ROOT)
print(f'found {len(items)} stereo pairs under {DATA_ROOT}')
assert len(items) > 100, 'Expected thousands of pairs — check DATA_ROOT.'
for lp, rp, pp in items[:3]:
    print(' ', lp.replace(DATA_ROOT, '<data>'))


## 7. Train (DDP via `torchrun`)

On T4 x2 this runs `--nproc_per_node=2`; on a single P100 it falls back to one process. Logs stream live; each epoch ends with a VAL EPE line and checkpoint save.

In [ ]:
import torch, os, subprocess, shlex

N_GPU = torch.cuda.device_count()
launcher = f'torchrun --standalone --nproc_per_node={N_GPU}' if N_GPU > 1 \
           else 'python3'

cmd = (
    f'{launcher} /kaggle/working/train_ddp.py '
    f'--data_root {shlex.quote(DATA_ROOT)} '
    f'--out_dir /kaggle/working '
    f'--epochs {EPOCHS} --batch {BATCH_PER_GPU} '
    f'--inf_h {INF_H} --inf_w {INF_W} '
    f'--lr {LR_PEAK} --random_erase_p {RANDOM_ERASE} '
    f'--num_workers 2 --log_every 50'
)
print('launching:\n', cmd, '\n')
rc = subprocess.call(shlex.split(cmd))
print(f'\nexit code: {rc}')
assert rc == 0, 'training failed — check logs above'


## 8. Export best checkpoint to all formats

In [ ]:
import subprocess, shlex
cmd = (
    f'python3 /kaggle/working/export_v8.py '
    f'--ckpt /kaggle/working/stereolite_v8_best.pth '
    f'--out_dir /kaggle/working '
    f'--inf_h {INF_H} --inf_w {INF_W}'
)
print(cmd)
subprocess.check_call(shlex.split(cmd))


## 8b. GPU inference demo — load each format and time it

Both `.pth` and `.ts.pt` are PyTorch-runnable on GPU. The `.pth` is a plain state_dict (needs the source code at import time); the `.ts.pt` is a self-contained TorchScript — no source needed to load and run.


In [ ]:
import torch, time, sys, os
sys.path.insert(0, '/kaggle/working/src')
os.environ.setdefault('XFORMERS_DISABLED', '1')
from d1_tile import StereoLite, StereoLiteConfig

device = torch.device('cuda')
L = torch.rand(1, 3, INF_H, INF_W, device=device) * 255
R = torch.rand(1, 3, INF_H, INF_W, device=device) * 255

def bench(model, n=30, warmup=5):
    with torch.no_grad():
        for _ in range(warmup): model(L, R)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n): model(L, R)
        torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000

# (A) .pth — state_dict + source code
m_pth = StereoLite(StereoLiteConfig(backbone='mobilenet', use_dav2=False)).to(device)
ck = torch.load('/kaggle/working/stereolite_v8_best.pth',
                 map_location=device, weights_only=False)
m_pth.load_state_dict(ck['model'])
m_pth.eval()
ms_pth = bench(m_pth)
print(f'.pth (state_dict) on GPU: {ms_pth:.1f} ms  '
      f'params={sum(p.numel() for p in m_pth.parameters())/1e6:.3f} M')

# (B) .ts.pt — TorchScript (no source needed)
m_ts = torch.jit.load('/kaggle/working/stereolite_v8.ts.pt').to(device).eval()
ms_ts = bench(m_ts)
print(f'.ts.pt (TorchScript) on GPU: {ms_ts:.1f} ms')

# Parity check
with torch.no_grad():
    a = m_pth(L, R).cpu().numpy()
    b = m_ts(L, R).cpu().numpy()
import numpy as np
print(f'|.pth - .ts.pt| max: {np.abs(a - b).max():.5f}  '
      f'mean: {np.abs(a - b).mean():.6f}')

del m_pth, m_ts
torch.cuda.empty_cache()


## 9. List artifacts + sizes

In [ ]:
import os
print(f'{"file":<30s}  size')
print('-' * 42)
for f in sorted(os.listdir('/kaggle/working')):
    path = os.path.join('/kaggle/working', f)
    if os.path.isfile(path):
        sz = os.path.getsize(path)
        print(f'{f:<30s}  {sz/1e6:8.2f} MB')


## 10. Bundle for download

Zip includes:
- model artifacts (.pth, .ts.pt, .onnx x 3)
- `src/` — the exact Python source files needed to load the `.pth` on your own machine. Just `sys.path.insert(0, 'src')` and `from d1_tile import StereoLite, StereoLiteConfig`.
- `load_pth_example.py` — minimal loader / inference script
- `train_log.csv` — training curves


In [ ]:
import zipfile, os
out = '/kaggle/working/stereolite_v8_artifacts.zip'
MODEL_FILES = [
    'stereolite_v8_best.pth',
    'stereolite_v8_last.pth',
    'stereolite_v8.ts.pt',
    'stereolite_v8_fp32.onnx',
    'stereolite_v8_fp16.onnx',
    'stereolite_v8_int8.onnx',
    'train_log.csv',
]
SRC_FILES = [
    'src/_blocks.py',
    'src/d1_tile/__init__.py',
    'src/d1_tile/model.py',
    'src/d1_tile/tile_propagate.py',
    'src/sceneflow_loader.py',
]

LOADER_EXAMPLE = '''"""Minimal loader for stereolite_v8_best.pth.

Run:  python3 load_pth_example.py path/to/left.png path/to/right.png
Outputs disparity as a TURBO-coloured PNG next to the left image.
"""
import sys, os, numpy as np, torch, cv2
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "src"))
from d1_tile import StereoLite, StereoLiteConfig

CKPT = os.path.join(os.path.dirname(__file__), "stereolite_v8_best.pth")
INF_H, INF_W = 384, 768

def main(left_path, right_path):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    m = StereoLite(StereoLiteConfig(backbone="mobilenet", use_dav2=False)).to(dev)
    ck = torch.load(CKPT, map_location=dev, weights_only=False)
    m.load_state_dict(ck["model"])
    m.eval()
    L = cv2.resize(cv2.imread(left_path), (INF_W, INF_H))
    R = cv2.resize(cv2.imread(right_path), (INF_W, INF_H))
    Lt = torch.from_numpy(cv2.cvtColor(L, cv2.COLOR_BGR2RGB)).float().permute(2,0,1).unsqueeze(0).to(dev)
    Rt = torch.from_numpy(cv2.cvtColor(R, cv2.COLOR_BGR2RGB)).float().permute(2,0,1).unsqueeze(0).to(dev)
    with torch.no_grad():
        d = m(Lt, Rt).squeeze().cpu().numpy()
    lo, hi = float(np.percentile(d, 2)), float(np.percentile(d, 98))
    v = np.clip((d - lo) / max(hi - lo, 1e-6), 0, 1) * 255
    out = cv2.applyColorMap(v.astype(np.uint8), cv2.COLORMAP_TURBO)
    op = os.path.splitext(left_path)[0] + "_disp.png"
    cv2.imwrite(op, out)
    print(f"wrote {op}  (disparity range {d.min():.2f} .. {d.max():.2f})")

if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])
'''

with open('/kaggle/working/load_pth_example.py', 'w') as f:
    f.write(LOADER_EXAMPLE)

with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in MODEL_FILES:
        p = os.path.join('/kaggle/working', f)
        if os.path.exists(p):
            z.write(p, arcname=f)
            print(f'  + {f} ({os.path.getsize(p)/1e6:.1f} MB)')
        else:
            print(f'  ! missing: {f}')
    for f in SRC_FILES:
        p = os.path.join('/kaggle/working', f)
        if os.path.exists(p):
            z.write(p, arcname=f)
            print(f'  + {f} ({os.path.getsize(p)/1024:.1f} KB)')
    z.write('/kaggle/working/load_pth_example.py', arcname='load_pth_example.py')
    print(f'  + load_pth_example.py')
    # Iterative sample panels (progress snapshots)
    samples_dir = '/kaggle/working/samples'
    if os.path.isdir(samples_dir):
        count = 0
        for f in sorted(os.listdir(samples_dir)):
            p = os.path.join(samples_dir, f)
            z.write(p, arcname=f'samples/{f}')
            count += 1
        print(f'  + samples/ ({count} iterative panels)')

print(f'\nwrote {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
print('Download from the Kaggle notebook Output panel.')


---

### Notes
- **Expected artifact sizes** (on a trained v8 checkpoint, ~2.14 M params):
  - `stereolite_v8_best.pth` ~9 MB (fp32 state_dict)
  - `stereolite_v8.ts.pt` ~9 MB (TorchScript traced)
  - `stereolite_v8_fp32.onnx` ~15 MB (opset 17 + unrolled loops)
  - `stereolite_v8_fp16.onnx` ~8 MB (halved weights)
  - `stereolite_v8_int8.onnx` ~75 MB (large because the int8 preprocessor un-folds our 24-step cost-volume and 8-iter refinement loops; only conv/matmul weights are int8, activations stay fp32). For smaller int8 artifacts use static quantization with a calibration set.
- **Exporter backend**: we force `dynamo=False` (legacy tracer) because the new dynamo exporter currently mis-names initializers when a model mixes timm BatchNorm (inside MobileNetV2) with GroupNorm (in our blocks) and produces a graph that onnxruntime refuses to load.
- **If training OOMs**: reduce `BATCH_PER_GPU` to 2, or `INF_H/INF_W` to 320/640.
- **Resume**: re-run the training cell with an additional `--resume /kaggle/working/stereolite_v8_last.pth` argument if the session restarted mid-run.
- **Single-GPU fallback**: if you switch to P100 (1 GPU), the launcher automatically drops DDP and runs single-process.